In [61]:
from flax import nnx
import jax
import jax.numpy as jnp
import pandas as pd
import glob
from pathlib import Path
import json
import os
import sys
import shlex
from collections import defaultdict
sys.path.append("..")

from qnrl.algorithms.algorithm import Algorithm

import wandb
api = wandb.Api()

In [62]:
root_sweeps = Path("../../wandb-sweep-data-qnrl/")
root_sweeps.mkdir(parents=True, exist_ok=True)

In [63]:
project_id = "qnrl"

In [64]:
# List of sweep names and their IDs for reference.
sweep_name_id = { # name: id
    # QnRL with QFT.
    'qnrl/FrozenLake-v1': 'sgi6zb0t',
    'qnrl/CliffWalking-v1': 'h4g5er22',
    'qnrl/Acrobot-v1': 'sd4va92r',
    'qnrl/CartPole-v1': '4ykq4hvw',
    
    # # QnRL without QFT.
    # 'qnrl-no-qft/FrozenLake-v1': 'y7a79v7c',
    # 'qnrl-no-qft/CliffWalking-v1': 'winn4pzr',
    # 'qnrl-no-qft/Acrobot-v1': 'b7dncod3',
    # 'qnrl-no-qft/CartPole-v1': 'a09u7b83',
    
    # C51.
    'c51/FrozenLake-v1': '3exa9jvo',
    'c51/CliffWalking-v1': '1nlfftsy',
    'c51/Acrobot-v1': 's3xwn85k',
    'c51/CartPole-v1': '7raha9om',
    
    # Epsilon going to zero.
    'qnrl/Acrobot-v1_eps-end=0': 'o4crwlxv',
    'qnrl/CartPole-v1_eps-end=0': 'nj16ychj',
    'qnrl/CliffWalking-v1_eps-end=0': '1om3jymb',
    'qnrl/FrozenLake-v1_eps-end=0': '8c4ggf0g',
    # 
    'c51/Acrobot-v1_eps-end=0': 'dkgnfcnp',
    'c51/CartPole-v1_eps-end=0': '1hwjmjg5',
    'c51/CliffWalking-v1_eps-end=0': '2p7uum4y',
    'c51/FrozenLake-v1_eps-end=0': 'o92xk1kc',
    
    
    # Total timesteps 200_000.
    'qnrl/FrozenLake-v1_total-timesteps=200000': 'sr2l6cra',
    'qnrl/CliffWalking-v1_total-timesteps=200000': 'm4mmjru4',
    'qnrl/Acrobot-v1_total-timesteps=200000': 'ljy4rhrs',
    'qnrl/CartPole-v1_total-timesteps=200000': 'ak9sbuph',
    # 
    'c51/FrozenLake-v1_total-timesteps=200000': '0q44xa91',
    'c51/CliffWalking-v1_total-timesteps=200000': 'ktogmnn8',
    'c51/Acrobot-v1_total-timesteps=200000': 'jlyv3bjp',
    'c51/CartPole-v1_total-timesteps=200000': 'qv6vqixl',
    
    
    # Total timesteps 500_000.
    'qnrl/FrozenLake-v1_total-timesteps=500000': 'rgmgrgqt',
    'qnrl/CliffWalking-v1_total-timesteps=500000': '196jssk4',
    'qnrl/Acrobot-v1_total-timesteps=500000': 'j5xs5362',
    'qnrl/CartPole-v1_total-timesteps=500000': 'mdjr7pk4',
    # 
    'c51/FrozenLake-v1_total-timesteps=500000': 'uu9ap7zs',
    'c51/CliffWalking-v1_total-timesteps=500000': 'jyc7e6fc',
    'c51/Acrobot-v1_total-timesteps=500000': 'usa8i3zg',
    'c51/CartPole-v1_total-timesteps=500000': 'd0vei6vi',
    
    
    # Breakout-v5.
    # 'qnrl/Breakout-v5': 'i48yfp4o',
    'qnrl/Breakout-v5': 'yavb5d6j',
    # 'c51/Breakout-v5': 'sg45do2b',
    'c51/Breakout-v5': 'gxpmvyxv',
}

# Best runs tagged for each sweep.
# Retrieve via the code:
#    {sweep_name: [run.id for run in api.sweep(f"{project_id}/{sweep_id}").runs if "best-seed-group" in run.tags] for sweep_name, sweep_id in sweep_name_id.items()}
best_runs_for_sweep = { # name: [id]
    # QnRL.
    "qnrl/FrozenLake-v1": ["asqt9rhf", "pfhzc37a", "tcmzk822", "o8bvmgah"],
    "qnrl/CliffWalking-v1": ["gg17kbpu", "3vn1egt0", "j9n68hn4", "au4el5rn"],
    "qnrl/Acrobot-v1": ["5m2sdda0", "oehcc3xd", "40j6hb1a", "l1c029ff"],
    "qnrl/CartPole-v1": ["atroi3e3", "ridysmu3", "z5elw4vo", "inwzalpf"],
    
    # C51.
    "c51/FrozenLake-v1": ["3dow3k33", "69lp8uef", "ard9oh61", "6ozx435e"],
    "c51/CliffWalking-v1": ["5ti67g7h", "gbkzvg60", "bgrjs1z8", "nk5iyhn3"],
    # "c51/Acrobot-v1": ["5tai0d5b", "57wv5oow", "nqtj0noh", "6xnb70dg"], # max eval/episode_reward, but very large.
    "c51/Acrobot-v1": ["h4xq6mf0", "hlqi1kae", "36eorruy", "6sgyeyvq"], # next-best eval/episode_reward, but size more comparable to QnRL.
    # "c51/CartPole-v1": ["e0x5vmzu", "xxcaueu9", "jit954em", "x0tak5kj"], # max eval/episode_reward, but very large.
    "c51/CartPole-v1": ["7s948sos", "pcq3nv1k", "q3mylk2i", "4eh2yh8z"], # next-best eval/episode_reward, but size more comparable to QnRL.
    
    # Epsilon going to zero.
    'qnrl/Acrobot-v1_eps-end=0': ['zz3qy24i', '8i9dyq5f', 'an2jxj88', 'o90nxq69'],
    # 'qnrl/Acrobot-v1_eps-end=0': ['zz3qy24i', '8i9dyq5f'],
    'qnrl/CartPole-v1_eps-end=0': ['d8hfwn7e', 'kzwr21ur', 'x7gsy1b3', 'cjfuef2c'],
    'qnrl/CliffWalking-v1_eps-end=0': ['a576n2nr', 'e1xaiqif', 'nr38lnei', 'y3yr5ij6'],
    'qnrl/FrozenLake-v1_eps-end=0': ['jkkjr9ya', 'rv6qgu0k', 'paxwbjqq', '8o2v51bi'],
    # 
    'c51/Acrobot-v1_eps-end=0': ['7uzq8h7d', 'wtkt7le0', 'xslr64rg', 'kyj7sr44'],
    # 'c51/Acrobot-v1_eps-end=0': ['7uzq8h7d', 'wtkt7le0'],
    'c51/CartPole-v1_eps-end=0': ['81zh78yz', 'va1nvy9s', 'wswa8kj8', '9z6geld3'],
    'c51/CliffWalking-v1_eps-end=0': ['1to89k79', 'otp2fau2', '24ptavwa', '0hn2uki9'],
    'c51/FrozenLake-v1_eps-end=0': ['oull4gip', 'u8aizgya', '77erlxo7', '1ro376lv'],
    
    
    # Total timesteps 200_000.
    'qnrl/FrozenLake-v1_total-timesteps=200000': ['vwxhptgk', 'htg8fy7d', 'oyhaupnv', '848tggcd'],
    'qnrl/CliffWalking-v1_total-timesteps=200000': ['i975ifa4', 'ktwf5h1j', 'i0vsuaqe', 'u0jtk06p'],
    'qnrl/Acrobot-v1_total-timesteps=200000': ['m9kbl7ko', '3i7x4u6s', 'at54ujkf', 'zmcrolpt'],
    # 'qnrl/Acrobot-v1_total-timesteps=200000': ['m9kbl7ko', '3i7x4u6s', 'at54ujkf'],
    'qnrl/CartPole-v1_total-timesteps=200000': ['rfapohrr', 'wad6ukuh', '3sj4j2z9', 'f1qixvf1'],
    # 'qnrl/CartPole-v1_total-timesteps=200000': ['rfapohrr', '3sj4j2z9', 'f1qixvf1'],
    # 
    'c51/FrozenLake-v1_total-timesteps=200000': ['mo0kv45d', 'yhccsote', 'msw7wga2', 'vlcmakfu'],
    'c51/CliffWalking-v1_total-timesteps=200000': ['vsvf8jm7', '8awsayx8', 'oobmx45v', '5yy7ctc8'],
    'c51/Acrobot-v1_total-timesteps=200000': ['qrfcsc0a', '6ho8gp79', '73lunb5z', 'uloq2afw'],
    # 'c51/Acrobot-v1_total-timesteps=200000': ['qrfcsc0a', '6ho8gp79', '73lunb5z'],
    'c51/CartPole-v1_total-timesteps=200000': ['1u3egh58', '7dr3grar', 'l4p5wp1a', 'on10tcbl'],
    # 'c51/CartPole-v1_total-timesteps=200000': ['1u3egh58', 'l4p5wp1a', 'on10tcbl'],
    
    
    
    # Total timesteps 500_000.
    'qnrl/FrozenLake-v1_total-timesteps=500000': ['3ksb5d42', '7hx34c90', '6ucfuo8x', 'w9ed8k5e'],
    'qnrl/CliffWalking-v1_total-timesteps=500000': ['s7giq412', '705hzdeq', 'hmfrf5au', 'qni0e1w0'],
    'qnrl/Acrobot-v1_total-timesteps=500000': ['7qhyg2fr', 'wk2dkr5c', '4twh6bqn', '3r9x0mff'],
    # 'qnrl/Acrobot-v1_total-timesteps=500000': ['7qhyg2fr', '4twh6bqn'],
    'qnrl/CartPole-v1_total-timesteps=500000': ['pg9isy9x', 'kxwyn7nq', 'gj3z0x8r', 'm8qpoub5'],
    # 
    'c51/FrozenLake-v1_total-timesteps=500000': ['3o58z9qy', '4pufcvf8', '0yycmhur', 'gzyzdaj2'],
    'c51/CliffWalking-v1_total-timesteps=500000': ['xpqgvf2m', '9u76311i', 'mw0v9iqd', 'wlxbs7hk'],
    'c51/Acrobot-v1_total-timesteps=500000': ['x2ju0t1u', 'b8tswyha', 'y1as92aq', 'jhxqg8ke'],
    # 'c51/Acrobot-v1_total-timesteps=500000': ['y1as92aq', 'jhxqg8ke'],
    'c51/CartPole-v1_total-timesteps=500000': ['j2jcldhn', 'mo76ja6b', '52jdh6k0', 'f8ndihoz'],
    
    
    # Breakout-v5.
    # 'qnrl/Breakout-v5': ['m6h00pt8', 'djze3jz3', 'x31uhzh7', '4po9lplb'],
    'qnrl/Breakout-v5': ['5lut1vib', 'hv40n10v', '7bvh3pmo', 'dqh0bsum'],
    # 'c51/Breakout-v5': ['0x0ds6k0', 'apixh9px', 'qfb090bx', 'myy3getf'],
    'c51/Breakout-v5': ['px8cxhah', '6n0rtly1', 'rvgscwo2', '1vod0zn8'],
}

In [65]:
import yaml
import ast
from qnrl.serialization import hash_serializable_object
from typing import Any

def get_run_config(config_file: Path):
    with open(str(config_file), 'r') as f:
        config = yaml.safe_load(f)
    return config

def get_run_metadata(metadata_file: Path):
    # Load metadata as JSON dictionary.
    with open(str(metadata_file), 'r') as f:
        metadata = json.load(f)
    return metadata

def get_run_args(metadata_file: Path) -> dict:
    metadata = get_run_metadata(metadata_file)
    l: list[tuple[str, str]] = [tokens.split('=', 1) for tokens in metadata['args'] if '--' in tokens and '=' in tokens]
    d = dict((key.replace('--', '', 1), ast.literal_eval(val)) for key, val in l)
    return d

def get_run_config_dict_from_metadata(metadata_file: Path) -> dict:
    """Compute run config hash from original run program command string."""
    arg_dict = get_run_args(metadata_file)
    return arg_dict['config']
    
    
    # metadata = get_run_metadata(metadata_file)
    
    # # Get config string from run program command arguments.
    # # Format is: --config="{...}"
    # cfg_str = metadata['args'][1].split("--config=")[1]
    # # Convert this string to dictionary via literal evaluation.
    # cfg_dict = ast.literal_eval(cfg_str)
    # return cfg_dict

In [66]:
from typing import Iterator

def eval(sweep_id: str, run_id: str, eval_kwargs_override: dict = {}) -> Iterator[dict]:
    
    dir_run = root_sweeps/f"sweep-{sweep_id}"/f"run-{run_id}"
    metadata_file = dir_run/"wandb-metadata.json"

    arg_dict = get_run_args(metadata_file)
    
    print(f"Evaluating: sweep={sweep_id}, run={run_id}, num_eval={arg_dict['num_eval']}")

    cfg = arg_dict['config']
    # cfg_hash, _ = hash_serializable_object(cfg) # Compute hash for config object.
    algo_config = Algorithm.from_dict(cfg)
    
    # Load parameters from model save.
    model_dirs = {Path(root).stem: Path(root) for root in glob.glob(str(dir_run/"models"/"*"))}
    algo_config.load(model_dirs)
    # nnx.display(nnx.state(algo_config.model))
    
    eval_reports = []
    eval_seed = arg_dict['eval_seed'] if arg_dict['eval_seed'] is not None else algo_config.seed
    key = jax.random.PRNGKey(seed=eval_seed)
    for eval_idx in range(arg_dict['num_eval']):
        
        # Generate a new random key.
        key, subkey = jax.random.split(key=key)
        
        # Evaluate the model using a different random seed.
        seed_eval = int(subkey[0])
        eval_kwargs = {}
        if 'eval_total_timesteps' in arg_dict and arg_dict['eval_total_timesteps'] is not None:
            eval_kwargs['total_timesteps'] = arg_dict['eval_total_timesteps']
        _, eval_report = algo_config.eval(seed=seed_eval, **{**eval_kwargs, **eval_kwargs_override})
        
        # Add an index counter for the evaluation report.
        eval_report['eval/index'] = eval_idx
        
        yield eval_report


def eval_via_env(env_key: str, eval_kwargs_override: dict = {}):
    eval_reports = defaultdict(lambda: defaultdict(list[dict])) # { sweep_id: { run_id: [report_0, report_1, ...] } }
    sweep_id = sweep_name_id[env_key]
    for run_id in best_runs_for_sweep[env_key]:
        eval_reports[sweep_id][run_id] = list(eval(sweep_id, run_id, eval_kwargs_override))
    return eval_reports

In [67]:
# env_key = "qnrl/CartPole-v1"
# env_key = "c51/CartPole-v1"

# env_key = "qnrl/FrozenLake-v1"
# env_key = "c51/FrozenLake-v1"


env_key = "qnrl/Acrobot-v1"
# env_key = "c51/Acrobot-v1"
report_dict = eval_via_env(env_key, eval_kwargs_override=dict(eps=0.001))

Evaluating: sweep=sd4va92r, run=5m2sdda0, num_eval=10


  0%|          | 163/100000 [00:02<24:21, 68.31step/s, eval/step=163, eval/episode_reward=-163, eval/episode_length=164, eval/episode_time=2.38]


Evaluating: sweep=sd4va92r, run=oehcc3xd, num_eval=10


  0%|          | 499/100000 [00:07<25:10, 65.86step/s, eval/step=499, eval/episode_reward=-500, eval/episode_length=500, eval/episode_time=7.57]


Evaluating: sweep=sd4va92r, run=40j6hb1a, num_eval=10


  0%|          | 97/100000 [00:01<26:16, 63.36step/s, eval/step=97, eval/episode_reward=-97, eval/episode_length=98, eval/episode_time=1.52]


Evaluating: sweep=sd4va92r, run=l1c029ff, num_eval=10


  0%|          | 93/100000 [00:01<26:25, 63.02step/s, eval/step=93, eval/episode_reward=-93, eval/episode_length=94, eval/episode_time=1.46]


In [68]:
nnx.display(report_dict)

In [69]:
def get_metric_as_series_for_run(metric_key: str, run_id: str, eval_reports: list[dict]):
    """Extract metric key as Pandas series and change metric key to row ID."""
    return pd.DataFrame(eval_reports)[metric_key].rename(run_id)

def get_metric_as_dataframe_for_sweep(metric_key: str, sweep_id: str):
    l = []
    for run_id, eval_reports in report_dict[sweep_id].items():
        # l.append(pd.DataFrame(eval_reports)['eval/episode_reward'].to_list())
        l.append(pd.DataFrame(eval_reports)[metric_key].rename(run_id))
    df = pd.concat(l, axis=1, join='outer') # column keys are 'row_id'.
    return df

In [70]:
for sweep_id in report_dict.keys():
    df = get_metric_as_dataframe_for_sweep('eval/episode_reward', sweep_id)
    print(df)
    print(df.mean())
    print(df.mean().mean())

   5m2sdda0  oehcc3xd  40j6hb1a  l1c029ff
0    -154.0    -500.0    -117.0    -112.0
1    -125.0    -136.0     -97.0     -69.0
2    -158.0    -500.0     -96.0     -70.0
3    -111.0    -500.0     -93.0     -99.0
4    -173.0    -500.0    -119.0     -81.0
5    -169.0    -159.0    -117.0     -96.0
6    -138.0    -500.0    -270.0     -70.0
7    -162.0    -500.0    -106.0     -69.0
8    -179.0    -500.0     -86.0     -76.0
9    -163.0    -500.0     -97.0     -93.0
5m2sdda0   -153.2
oehcc3xd   -429.5
40j6hb1a   -119.8
l1c029ff    -83.5
dtype: float64
-196.5
